# Building and integrating a testing framework

This notebook demonstrates how to use the testing framework to evaluate different LLM models.

## Models to Test
- gpt-5.2
- gpt-5-mini
- gpt-5-nano
- gpt-4.1-nano
- gpt-4o
- gpt-4o-mini
- gpt-4.1-mini

## Setup and Imports

In [ ]:
import sys
import os
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Add project root to path
project_root = os.path.dirname(os.getcwd())
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from sql_ai_agent.testing import TestRunner, get_test_cases
from sql_ai_agent.testing.test_cases import get_test_summary

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## View Test Cases

In [ ]:
# Get test summary
summary = get_test_summary()

print(f"Total Test Cases: {summary['total_tests']}")
print(f"\nBy Difficulty:")
for diff, count in sorted(summary['by_difficulty'].items()):
    print(f"  {diff.capitalize()}: {count}")

print(f"\nBy Category:")
for cat, count in sorted(summary['by_category'].items()):
    print(f"  {cat}: {count}")

In [ ]:
# Display all test cases
all_tests = get_test_cases()

test_df = pd.DataFrame([
    {
        'ID': test.id,
        'Difficulty': test.difficulty,
        'Category': test.category,
        'Question': test.question[:80] + '...' if len(test.question) > 80 else test.question
    }
    for test in all_tests
])

display(test_df)

## Initialize Test Runner

In [ ]:
# Initialize the test runner with database connection
runner = TestRunner(
    db_host="postgres",
    db_port=5432,
    db_name="my_db",
    db_user="postgres",
    db_password="password",
    table_name="air_traffic"
)

print("✅ Test runner initialized successfully")

## Define Models to Test

In [ ]:
# Define models to test
models_to_test = {
    'openai': [
        'gpt-5.2',
        'gpt-5-mini',
        'gpt-5-nano',
        'gpt-4.1-nano',
        'gpt-4o',
        'gpt-4o-mini',
        'gpt-4.1-mini'
    ]
}

print("Models to test:")
for provider, models in models_to_test.items():
    print(f"\n{provider.upper()}:")
    for model in models:
        print(f"  - {model}")

## Option 1: Run All Tests for All Models

This will run all 20 test cases + 5 debug tests for each model.

In [ ]:
# Run all tests for all models
# WARNING: This may take a while (20 tests + 5 debug tests per model)

print("Starting comprehensive test run...\n")
print("This will test:")
print(f"  - {len(models_to_test['openai'])} models")
print(f"  - 20 query generation tests per model")
print(f"  - 5 debug mechanism tests per model")
print(f"  - Total: {len(models_to_test['openai']) * 25} test executions\n")

results_df = runner.run_all_tests(
    providers=['openai'],
    models=models_to_test,
    max_debug_trials=3
)

print("\n✅ All tests completed!")
print(f"Total results: {len(results_df)} test executions")

## Save Results to CSV

In [ ]:
# Generate timestamp for file naming
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Save detailed results
results_file = f'test_results_{timestamp}.csv'
results_df.to_csv(results_file, index=False)
print(f"✅ Detailed results saved to: {results_file}")

# Generate and save summary
summary_df = runner.generate_summary_report(results_df)
summary_file = f'test_summary_{timestamp}.csv'
summary_df.to_csv(summary_file, index=False)
print(f"✅ Summary report saved to: {summary_file}")

## View Summary Results

In [ ]:
# Display summary report
print("\n" + "="*100)
print("TEST SUMMARY")
print("="*100 + "\n")

display(summary_df)

## Analyze Results by Test Type

In [ ]:
# Query Generation Results
query_results = results_df[results_df['test_type'] == 'query_generation'].copy()

print("\n" + "="*80)
print("QUERY GENERATION RESULTS")
print("="*80 + "\n")

query_summary = query_results.groupby('model').agg({
    'success': ['count', 'sum', 'mean'],
    'execution_time': 'mean'
}).round(3)

query_summary.columns = ['Total Tests', 'Successful', 'Success Rate', 'Avg Time (s)']
query_summary['Success Rate'] = (query_summary['Success Rate'] * 100).round(2).astype(str) + '%'

display(query_summary.sort_values('Successful', ascending=False))

In [ ]:
# Debug Mechanism Results
debug_results = results_df[results_df['test_type'] == 'debug_mechanism'].copy()

if not debug_results.empty:
    print("\n" + "="*80)
    print("DEBUG MECHANISM RESULTS")
    print("="*80 + "\n")
    
    debug_summary = debug_results.groupby('model').agg({
        'success': ['count', 'sum', 'mean'],
        'trials_needed': 'mean',
        'execution_time': 'mean'
    }).round(3)
    
    debug_summary.columns = ['Total Tests', 'Fixed', 'Fix Rate', 'Avg Trials', 'Avg Time (s)']
    debug_summary['Fix Rate'] = (debug_summary['Fix Rate'] * 100).round(2).astype(str) + '%'
    
    display(debug_summary.sort_values('Fixed', ascending=False))

## Analyze Results by Difficulty

In [ ]:
# Success rate by difficulty level
if 'difficulty' in query_results.columns:
    print("\n" + "="*80)
    print("SUCCESS RATE BY DIFFICULTY")
    print("="*80 + "\n")
    
    difficulty_pivot = pd.crosstab(
        query_results['model'],
        query_results['difficulty'],
        query_results['success'],
        aggfunc='mean'
    ).round(3) * 100
    
    # Reorder columns: easy, medium, hard
    column_order = [col for col in ['easy', 'medium', 'hard'] if col in difficulty_pivot.columns]
    difficulty_pivot = difficulty_pivot[column_order]
    
    display(difficulty_pivot.style.format("{:.2f}%"))

## Visualizations

In [ ]:
# Success Rate Comparison
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Prepare data
query_success = query_results.groupby('model')['success'].mean().sort_values(ascending=False) * 100
query_success_df = query_success.reset_index()
query_success_df.columns = ['model', 'success_rate']

# Create subplots with 1 row and 2 columns
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Query Generation Success Rate by Model', 'Debug Fix Rate by Model')
)

# Query Generation Success Rate (left subplot)
fig.add_trace(
    go.Bar(
        x=query_success_df['model'],
        y=query_success_df['success_rate'],
        name='Query Success',
        marker_color='steelblue',
        showlegend=False
    ),
    row=1, col=1
)

# Debug Fix Rate (right subplot)
if not debug_results.empty:
    debug_success = debug_results.groupby('model')['success'].mean().sort_values(ascending=False) * 100
    debug_success_df = debug_success.reset_index()
    debug_success_df.columns = ['model', 'fix_rate']
    
    fig.add_trace(
        go.Bar(
            x=debug_success_df['model'],
            y=debug_success_df['fix_rate'],
            name='Debug Fix',
            marker_color='coral',
            showlegend=False
        ),
        row=1, col=2
    )

# Update layout
fig.update_xaxes(title_text="Model", row=1, col=1)
fig.update_xaxes(title_text="Model", row=1, col=2)
fig.update_yaxes(title_text="Success Rate (%)", range=[0, 100], row=1, col=1)
fig.update_yaxes(title_text="Fix Rate (%)", range=[0, 100], row=1, col=2)

fig.update_layout(
    height=500,
    width=1200,
    showlegend=False,
    template='plotly_white'
)

fig.write_html(f'success_rates_{timestamp}.html')
fig.show()

In [ ]:
# Execution Time Comparison
import plotly.graph_objects as go

avg_times = query_results.groupby('model')['execution_time'].mean().sort_values()
avg_times_df = avg_times.reset_index()
avg_times_df.columns = ['model', 'avg_time']

fig = go.Figure()

fig.add_trace(go.Bar(
    x=avg_times_df['avg_time'],
    y=avg_times_df['model'],
    orientation='h',
    marker_color='seagreen',
    text=avg_times_df['avg_time'].round(3),
    textposition='auto',
))

fig.update_layout(
    title='Average Query Execution Time by Model',
    xaxis_title='Average Execution Time (seconds)',
    yaxis_title='Model',
    height=500,
    width=900,
    template='plotly_white',
    showlegend=False
)

fig.write_html(f'execution_times_{timestamp}.html')
fig.show()

In [ ]:
# Success Rate by Difficulty (Heatmap)
if 'difficulty' in query_results.columns:
    import plotly.graph_objects as go
    
    difficulty_pivot = pd.crosstab(
        query_results['model'],
        query_results['difficulty'],
        query_results['success'],
        aggfunc='mean'
    ) * 100
    
    # Reorder columns: easy, medium, hard
    column_order = [col for col in ['easy', 'medium', 'hard'] if col in difficulty_pivot.columns]
    difficulty_pivot = difficulty_pivot[column_order]
    
    # Create heatmap
    fig = go.Figure(data=go.Heatmap(
        z=difficulty_pivot.values,
        x=difficulty_pivot.columns,
        y=difficulty_pivot.index,
        colorscale='RdYlGn',
        zmin=0,
        zmax=100,
        text=difficulty_pivot.values.round(1),
        texttemplate='%{text:.1f}',
        textfont={"size": 12},
        colorbar=dict(title="Success Rate (%)")
    ))
    
    fig.update_layout(
        title='Success Rate by Model and Difficulty',
        xaxis_title='Difficulty Level',
        yaxis_title='Model',
        height=600,
        width=800,
        template='plotly_white'
    )
    
    fig.write_html(f'difficulty_heatmap_{timestamp}.html')
    fig.show()

## Detailed Failure Analysis

In [ ]:
# Show failed query generation tests
failed_queries = query_results[query_results['success'] == False].copy()

if not failed_queries.empty:
    print("\n" + "="*80)
    print(f"FAILED QUERY TESTS ({len(failed_queries)} failures)")
    print("="*80 + "\n")
    
    failure_summary = failed_queries.groupby(['model', 'test_id']).size().reset_index(name='count')
    
    print("Failures by Model and Test ID:")
    display(failure_summary.pivot(index='test_id', columns='model', values='count').fillna(0).astype(int))
    
    # Show details of failed tests
    print("\nFailed Test Details:")
    failed_details = failed_queries[[
        'model', 'test_id', 'question', 'difficulty', 'category', 
        'error_message', 'validation_message'
    ]].head(10)
    display(failed_details)
else:
    print("\n🎉 All query generation tests passed!")

In [ ]:
# Show failed debug tests
if not debug_results.empty:
    failed_debug = debug_results[debug_results['success'] == False].copy()
    
    if not failed_debug.empty:
        print("\n" + "="*80)
        print(f"FAILED DEBUG TESTS ({len(failed_debug)} failures)")
        print("="*80 + "\n")
        
        failed_debug_details = failed_debug[[
            'model', 'test_id', 'description', 'error_type', 
            'trials_needed', 'final_error'
        ]]
        display(failed_debug_details)
    else:
        print("\n🎉 All debug tests passed!")

## Model Comparison Summary

In [ ]:
# Create comprehensive comparison table
comparison_data = []

for model in models_to_test['openai']:
    model_query = query_results[query_results['model'] == model]
    model_debug = debug_results[debug_results['model'] == model] if not debug_results.empty else pd.DataFrame()
    
    row = {
        'Model': model,
        'Query Tests': len(model_query),
        'Query Success': model_query['success'].sum(),
        'Query Success %': f"{(model_query['success'].mean() * 100):.2f}%" if len(model_query) > 0 else 'N/A',
        'Avg Query Time (s)': f"{model_query['execution_time'].mean():.3f}" if len(model_query) > 0 else 'N/A',
        'Debug Tests': len(model_debug),
        'Debug Fixed': model_debug['success'].sum() if len(model_debug) > 0 else 0,
        'Debug Fix %': f"{(model_debug['success'].mean() * 100):.2f}%" if len(model_debug) > 0 else 'N/A',
        'Avg Trials': f"{model_debug[model_debug['success'] == True]['trials_needed'].mean():.2f}" if len(model_debug[model_debug['success'] == True]) > 0 else 'N/A'
    }
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*120)
print("COMPREHENSIVE MODEL COMPARISON")
print("="*120 + "\n")

display(comparison_df)

# Save comparison
comparison_df.to_csv(f'model_comparison_{timestamp}.csv', index=False)
print(f"\n✅ Model comparison saved to: model_comparison_{timestamp}.csv")

## Key Findings Summary

In [ ]:
print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80 + "\n")

# Best performing model for query generation
best_query_model = query_results.groupby('model')['success'].mean().idxmax()
best_query_rate = query_results.groupby('model')['success'].mean().max() * 100
print(f"🏆 Best Query Generation Model: {best_query_model} ({best_query_rate:.2f}% success rate)")

# Fastest model
fastest_model = query_results.groupby('model')['execution_time'].mean().idxmin()
fastest_time = query_results.groupby('model')['execution_time'].mean().min()
print(f"⚡ Fastest Model: {fastest_model} ({fastest_time:.3f}s avg execution time)")

# Best debug model
if not debug_results.empty:
    best_debug_model = debug_results.groupby('model')['success'].mean().idxmax()
    best_debug_rate = debug_results.groupby('model')['success'].mean().max() * 100
    print(f"🔧 Best Debug Model: {best_debug_model} ({best_debug_rate:.2f}% fix rate)")

# Overall statistics
total_tests = len(results_df)
total_success = results_df['success'].sum()
overall_rate = (total_success / total_tests * 100) if total_tests > 0 else 0
print(f"\n📊 Overall Statistics:")
print(f"   Total Tests: {total_tests}")
print(f"   Total Successful: {total_success}")
print(f"   Overall Success Rate: {overall_rate:.2f}%")

print("\n" + "="*80)